# Set up the isolated scientific notebook kernel

Setup is explicit; existing assets are verified before reuse.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from pathlib import Path
import hashlib,json,os,subprocess,sys
ROOT=TRACE_ROOT
INSTALL=False
RUNTIME_PYTHON=Path(os.environ.get('TRACE_LAB_RUNTIME_PYTHON',ROOT/'outputs/environments/runtime-verified/bin/python')).resolve()
OVERLAY=ROOT/'outputs/environments/scientific-kernel'
WHEELS=ROOT/'assets/package_cache/scientific_kernel'
manifest=json.loads((ROOT/'environment/scientific_kernel/artifact_manifest.json').read_text())
for row in manifest:
    path=ROOT/row['path']
    assert hashlib.sha256(path.read_bytes()).hexdigest()==row['sha256'],path
if INSTALL:
    if OVERLAY.exists():raise FileExistsError('Existing kernel overlay; verify it instead of overwriting')
    subprocess.run([str(RUNTIME_PYTHON),'-m','pip','install','--no-index','--find-links',str(WHEELS),'--no-deps','--require-hashes','--target',str(OVERLAY),'-r',str(ROOT/'environment/scientific_kernel/requirements.lock.txt')],check=True)
assert OVERLAY.is_dir(),'Set INSTALL=True after supplying the recorded wheels'
probe="import json,sys,numpy,torch,sklearn,IPython,ipykernel;from importlib.metadata import distributions;print(json.dumps({'python':sys.version,'numpy':numpy.__version__,'torch':torch.__version__,'sklearn':sklearn.__version__,'ipython':IPython.__version__,'ipykernel':ipykernel.__version__}))"
env={**os.environ,'PYTHONPATH':str(OVERLAY),'PYTHONDONTWRITEBYTECODE':'1'}
result=subprocess.run([str(RUNTIME_PYTHON),'-B','-c',probe],env=env,check=True,text=True,capture_output=True)
versions=json.loads(result.stdout.strip().splitlines()[-1]);assert versions['numpy']=='1.23.1' and versions['torch']=='2.7.1+cu118' and versions['sklearn']=='1.3.0'
from importlib.metadata import distributions
actual={d.metadata['Name'].lower().replace('_','-'):d.version for d in distributions(path=[str(OVERLAY)])}
expected={r['name'].lower().replace('_','-'):r['version'] for r in manifest};assert actual==expected
kernel=ROOT/'.venv/share/jupyter/kernels/trace-lab-scientific';kernel.mkdir(parents=True,exist_ok=True)
spec={'argv':[str(RUNTIME_PYTHON),'-B','-m','ipykernel_launcher','-f','{connection_file}'],'display_name':'Trace Lab scientific','language':'python','env':{'PYTHONPATH':str(OVERLAY),'PYTHONDONTWRITEBYTECODE':'1','MPLBACKEND':'Agg','MPLCONFIGDIR':str(ROOT/'outputs/.matplotlib'),'IPYTHONDIR':str(ROOT/'outputs/.ipython')}}
(kernel/'kernel.json').write_text(json.dumps(spec,indent=2)+'\n')
print('Verified',len(actual),'isolated tooling packages; registered',kernel.relative_to(ROOT))
print(json.dumps(versions,indent=2))


Verified 29 isolated tooling packages; registered .venv/share/jupyter/kernels/trace-lab-scientific
{
  "python": "3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12) \n[GCC 13.3.0]",
  "numpy": "1.23.1",
  "torch": "2.7.1+cu118",
  "sklearn": "1.3.0",
  "ipython": "7.34.0",
  "ipykernel": "6.29.5"
}
